# Test 2 (exploratory): disparity inside family F, architecture weight 0

All computing is in `p1_tests.py`, which imports `ws_core.py` (the code of the supplement). **Run All** reads the data again: changed data give new folders under `results_p1/`, unchanged data resume and only display. Results are reported, not read as pass or fail.

Weights: the 15 weights of Table A.11 as one flat list (values as the thesis code holds them: 1/45 and 1/3 where the table prints 0.022 and 0.333). Draws: `results_p1/weight_draws_1000.csv`, 1 000 rows of 15 factors exp(N(0, 0.35²)), seed 42 (log in `weight_draws_1000.json`). Band: 10 000 shuffles per run, seed 42, the same shuffles in every run, edge = 2.5th percentile; margin = band edge − ΔQ; holds = margin > 0.

Designs: family F only = every Lift + Cruise design plus every CVT* design (CVT with a tilting propulsor set and no tilting wing panel), `frames.family_mask`, the thesis's own definition. All-designs base. ΔQ = Q of the F designs of the cohort − mean Q of F's two earliest cohorts. Band: the cohort labels shuffled among the F designs only, ΔQ and F's baseline recomputed in each shuffle. Weights: Table A.11 with architecture 0; then the 1 000 draws of `weight_draws_1000.csv` with architecture kept at 0 (its factor ignored), band rebuilt in each draw.

In [ ]:
%matplotlib inline
import pandas as pd, ws_core as ws, p1_tests as p1
pd.set_option('display.width', 250, 'display.max_columns', 40, 'display.max_colwidth', 70, 'display.float_format', lambda x: f'{x:.4f}')
DATASETS = ('v1', 'thesis')    # eVTOL_dataset_v1 as it is now, then the thesis as delivered
P = {d: p1.prepare(d) for d in DATASETS}   # reads each dataset with the thesis code; unchanged data are not rewritten

## 1. F designs per cohort
The run stops here if the two earliest cohorts hold fewer than 20 F designs together.

In [ ]:
for d, p in P.items():
    print(d); display(p1.f_counts(p))

## 2. Check against the thesis code
With the Table A.11 weights (architecture included), the thesis function `cohesion_band` on the F designs alone against this code on the same 1 000 shuffles.

In [ ]:
for d, p in P.items():
    print(d); display(p1.baseline_checks(p).query("inputs.str.startswith('F')", engine='python'))

## 3. Hardware and timing

In [ ]:
T = {d: p1.timing(p) for d, p in P.items()}
T['v1']

## 4. Runs

In [ ]:
for d, p in P.items():
    print(f'================ {d}')
    p1.run_test2(p, backend=T[d].loc[1, 'used'])

## 5. Results
`test2_summary.csv`: per cohort, F designs, ΔQ and band edge with architecture 0 (and the edge from the first 1 000 shuffles beside it), the share of draws in which F's ΔQ is below its band with the Wilson 95 % interval, and the margin of the draws. `test2_draws.csv`: one row per draw and cohort.

In [ ]:
S2 = {d: p1.summary_test2(p) for d, p in P.items()}
S2['v1']

In [ ]:
S2['thesis']

## 6. Exploratory additions (author's choice of 2026-11-22, made after sections 1 to 5 were run)

Three readings of family F, reported as exploratory. Same designs (F only), same 10 000 shuffles (seed 42) and the same weight draws as above.

### 6a. Contraction since the peak cohort
ΔQ = Q(F, 2020-23) − Q(F, 2016-19), the cohort where F's Q is highest. Band: the cohort labels of the F designs of those two cohorts shuffled; edge = 2.5th percentile; holds = ΔQ below the edge. Rows: Table A.11 with architecture 0, Table A.11, each dimension alone; then the 1 000 draws with architecture 0. Files: `test2_explore_since_peak.csv`, `test2_explore_since_peak_draws.csv`.

In [ ]:
E = {d: p1.f_since_peak(p) for d, p in P.items()}
display(E['v1'][0], E['v1'][1])

In [ ]:
display(E['thesis'][0], E['thesis'][1])

### 6b. Lift + Cruise against CVT*, per cohort (descriptive, no band)
Table A.11 with architecture 0. File: `test2_explore_slc_cvt.csv`.

In [ ]:
{d: display(p1.f_slc_cvt(p)) for d, p in P.items()};

### 6c. Each dimension alone, against F's two earliest cohorts
The distance is then that dimension; the band as in section 4. File: `test2_explore_per_dimension.csv`.

In [ ]:
{d: display(p1.f_per_dimension(p)) for d, p in P.items()};